# E03 - ABLATION TIỀN XỬ LÝ: LÀM SẠCH, TEENCODE & TÁCH TỪ PYVI

**Câu hỏi (từ bài review mục 7):**
1. Các bước trong `src/text_cleaner.py` (emoji, teencode đơn ký tự `m/v/r/z`, `kh/cam`, tách từ `pyvi`) đóng góp bao nhiêu điểm F1 so với giữ câu thô?
2. Audit bằng mắt các câu bị biến đổi bởi teencode để xem tỷ lệ đổi sai nghĩa.

**Cố định:** cùng TF-IDF + LogisticRegression (Sentiment & Multi-label Aspect), trên cùng tập dòng đã dedup của `train_clean.csv` và `dev_clean.csv`.

In [1]:
import os, sys, re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score

ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'experiments' else os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
PROC_DIR = os.path.join(ROOT, 'data', 'processed')
from src.text_cleaner import segment_words
from src.exp_log import log_experiment

In [2]:
df_train = pd.read_csv(os.path.join(PROC_DIR, 'train_clean.csv'))
df_dev = pd.read_csv(os.path.join(PROC_DIR, 'dev_clean.csv'))

# Tạo thêm biến thể: câu thô chỉ tách từ pyvi (bỏ clean_text)
df_train['raw_seg'] = df_train['sentence'].fillna('').apply(segment_words)
df_dev['raw_seg'] = df_dev['sentence'].fillna('').apply(segment_words)

variants = [
    ('1_raw_only',      'sentence'),
    ('2_clean_no_pyvi', 'sentence_clean'),
    ('3_raw_plus_pyvi', 'raw_seg'),
    ('4_clean_and_pyvi','sentence_seg'),
]

## 1. SO SÁNH 4 BIẾN THỂ TRÊN CÙNG MODEL

In [3]:
TOKEN_PAT = r'(?u)\b\w+\b|\[\w+\]'
asp_cols = [f'asp_{i}' for i in range(5)]
rows = []
for name, col in variants:
    X_tr = df_train[col].fillna('')
    X_dev = df_dev[col].fillna('')
    
    p_sent = Pipeline([
        ('tfidf', TfidfVectorizer(token_pattern=TOKEN_PAT, ngram_range=(1, 2), min_df=2, max_df=0.95, max_features=15000, sublinear_tf=True)),
        ('clf', LogisticRegression(C=1.0, class_weight='balanced', max_iter=1000, random_state=42)),
    ]).fit(X_tr, df_train['sentiment'])
    sent_f1 = f1_score(df_dev['sentiment'], p_sent.predict(X_dev), average='macro')
    
    p_asp = Pipeline([
        ('tfidf', TfidfVectorizer(token_pattern=TOKEN_PAT, ngram_range=(1, 2), min_df=2, max_df=0.95, max_features=15000, sublinear_tf=True)),
        ('clf', OneVsRestClassifier(LogisticRegression(C=2.0, class_weight='balanced', max_iter=1000, random_state=42))),
    ]).fit(X_tr, df_train[asp_cols].values)
    asp_f1 = f1_score(df_dev[asp_cols].values, p_asp.predict(X_dev), average='macro')
    
    rows.append({'variant': name, 'column': col, 'dev_sent_f1': round(sent_f1, 4), 'dev_asp_ml_f1': round(asp_f1, 4)})
    log_experiment('E03', name, 'Text cleaning & pyvi ablation', {'column': col},
                   dev_f1=sent_f1, notes=f'dev_asp_ml_f1={asp_f1:.4f}')

pd.DataFrame(rows)

,variant,column,dev_sent_f1,dev_asp_ml_f1
0,1_raw_only,sentence,0.7418,0.8926
1,2_clean_no_pyvi,sentence_clean,0.7368,0.8927
2,3_raw_plus_pyvi,raw_seg,0.7395,0.8912
3,4_clean_and_pyvi,sentence_seg,0.7478,0.8911


## 2. AUDIT BẰNG MẮT CÁC CÂU BỊ TEENCODE RỦI RO THAY ĐỔI (`m/v/r/z/kh/cam/⭐`)

In [4]:
risk_pat = re.compile(r'(?<!\w)(?:m|v|r|z|kh|cam)(?!\w)|⭐', re.IGNORECASE)
changed = df_train[
    df_train['sentence'].fillna('').str.contains(risk_pat)
    & (df_train['sentence'].str.lower() != df_train['sentence_clean'].str.lower())
][['sentence', 'sentence_clean']]
print('Số câu chứa token rủi ro và bị biến đổi:', len(changed))
changed.sample(min(25, len(changed)), random_state=42)

Số câu chứa token rủi ro và bị biến đổi: 649


,sentence,sentence_clean
7535,Độ sáng màn hình hiển thị ngoài trời kém. Sau ...,Độ sáng màn hình hiển thị ngoài trời kém. Sau ...
2787,"Sp quá tuyệt vời, mọi thứ đều ok không còn gì ...","Sản phẩm quá tuyệt vời, mọi thứ đều ok không c..."
7076,"Máy ổn, \nhút wifi tốt, pin trâu, game khá mượ...","Máy ổn, hút wifi tốt, pin trâu, game khá mượt...."
5115,Cái gì cũng tạm được chỉ có pin tụt nh...,Cái gì cũng tạm được chỉ có pin tụt nhanh quá....
833,"Sản phẩm tốt , dùng oke, cam chụp siêu đẹp\nPi...","Sản phẩm tốt , dùng ok, camera chụp siêu đẹp P..."
5349,Đúng như tvấn nha k nói điêu chơi game cực tốt...,Đúng như tvấn nha không nói điêu chơi game cực...
2267,"M mua dịp sinh nhật của tgdđ , m mua con nà...","Mình mua dịp sinh nhật của thế giới di động , ..."
1629,Mới mua được 1 ngày mọi thứ điều thấy ok từ mà...,Mới mua được 1 ngày mọi thứ điều thấy ok từ mà...
2905,"Lúc đầu mua về mọi thứ ok, cam trc video call ...","Lúc đầu mua về mọi thứ ok, camera trước video ..."
3396,Mới vừa mua h ko hài lòng vân tay ko nhạy màn ...,Mới vừa mua h không hài lòng vân tay không nhạ...


## 3. KẾT LUẬN

- **Đóng góp tổng hợp của `clean_text` + `pyvi` trên Dev (`TF-IDF + LogisticRegression`):**
  - `4_clean_and_pyvi` đạt **`dev_sent_f1 = 0.7478`**, cao hơn cả bản thô (`1_raw_only = 0.7418`), chỉ làm sạch không tách từ (`2_clean_no_pyvi = 0.7368`), và chỉ tách từ không làm sạch (`3_raw_plus_pyvi = 0.7395`). Nhánh Aspect giữ ổn định quanh `0.8911–0.8927`.
- **Audit 25/649 câu bị token rủi ro tác động (`m/v/r/z/kh/cam/⭐`):** Đa số chuẩn hoá chính xác ngữ cảnh (`sp → sản phẩm`, `ko/k/hk → không`, `đc/dk → được`, `r → rồi`, `m → mình`, `cam → camera`, `😑 → thất_vọng`), chỉ còn `2/25` ca biên hi hữu đổi hơi gượng (`Ko có z → Không có vậy` ở dòng 7076; `Cam ơn → Camera ơn` ở dòng 631), đủ an toàn để giữ trong `src/text_cleaner.py`.